In [2]:
import pandas as pd
import requests
import os
import glob
import datetime as datetime
import pandas as pd
from dataIO import dataloader, webservices
from statisticscalculator import generalstatistics, climatestatistics
from plot_collection import stackedlineplots, streamflow_stat_plots, streamflow_stat_plots_matplotlib
import plotly.express as px

from aggregate_stats.aggregate_stats import calculate_annual_stats, aggregate_stats
from plot_collection.aggregate_plots import aggregate_mann_kendall_plots

In [3]:
headwater = pd.read_excel('headwater_sites.xlsx')
headwater.set_index('site_no', inplace=True)
site_list = list(headwater.index)
lats = list(headwater['lat'])
longs = list(headwater['long']) 

all_site_data = {}
for site in site_list[:4]:
    try:
        # basin = site[1][1]
        data = webservices.usgs_streamflow().get_data(start_date='1928-10-01', end_date='2023-10-01',sites=str(site)).reset_index()
        all_site_data[site] = data
    except Exception as e:
        print(e)
    finally:
        pass
    

https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12301250&startDT=1928-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12302055&startDT=1928-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12304500&startDT=1928-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060
https://waterservices.usgs.gov/nwis/dv/?format=json&sites=12306500&startDT=1928-10-01&endDT=2023-10-01&siteStatus=all&parameterCd=00060


In [4]:
import traceback

dates = {
    '1928-10-01':'2023-10-01', 
    # '1938-10-01':'1948-10-01', '1948-10-01':'1958-10-01', '1958-10-01':'1968-10-01', '1968-10-01':'1978-10-01',
    # '1978-10-01':'1988-10-01', '1988-10-01':'1998-10-01', '1998-10-01':'2008-10-01', '2008-10-01':'2018-10-01', '2018-10-01':'2023-10-01'
    # '1996-10-01':'2010-10-01'
}

# all_annual_stats_objects = {}
# for start_date, end_date in dates.items():
try:
    start_date = '1928-10-01'
    end_date = '2023-10-01'
    start_date = pd.to_datetime(start_date).date()
    end_date = pd.to_datetime(end_date).date()
    filtered_all_site_data = {}
    for site,df in all_site_data.items():
        filtered_all_site_data[site] = all_site_data.get(site)[all_site_data.get(site)['Date'].between(start_date, end_date)]

except Exception as e:
    print('Issue occurring when filtering data for time range')
    print(e)
    pass

    
annual_stats_objects = {}
for site, df in filtered_all_site_data.items():
    # try:
    # print(site)
    if df.empty:
        print(f'{site} has no data for the period between {start_date} and {end_date}.  Moving to next site.')
    else:
        d = dataloader.DataLoader(df, 'Date', name_of_Q_column='Discharge')
        print(d._df.head(1))
        s = climatestatistics.Streamflow(d)
        start_date = pd.to_datetime(start_date).date()
        end_date = pd.to_datetime(end_date).date()
        site_results = calculate_annual_stats()
        print(f's3: {site_results.s}')
        print(site_results.__dict__)
        site_results.calc_stats_headwater(stats_obj=s, site=site, start_date=start_date, end_date=end_date, headwater_metadata_df=headwater)

        annual_stats_objects[site] = site_results

    # all_annual_stats_objects[
            
        # except Exception as e:
        #     print('Failed importing data or running climate statistics.')
        #     print(e)
        #     traceback_str = ''.join(traceback.format_tb(e.__traceback__))
        #     print(traceback_str)
            



   index        Date  Discharge
0      0  2015-11-11       82.9


TypeError: __init__() missing 1 required positional argument: 'StreamflowClimateStatistics'

In [ ]:
s.calc_runoff_bw_days

In [ ]:
annual_stats_objects

In [ ]:
ammassed_results = aggregate_stats(annual_stats_objs=annual_stats_objects)
ammassed_results._aggregate_stats()
ammassed_results._convert_dicts_to_dfs()

In [ ]:
agg_plots = aggregate_mann_kendall_plots()
agg_plots.scatter(ammassed_results)
# agg_plots.save_scatter_figs(file_name='test_agg_plots.html', delete_existing=True)

In [ ]:
ammassed_results.__dict__

In [ ]:
s.threshold_vol_mann_kendall_test

In [ ]:
runoff_timing_mk_stats={}
runoff_timing_mk_stats['Threshold Vol MK Test'] = s.threshold_vol_mann_kendall_test
runoff_timing_mk_stats['Threshold Vol DOY MK Test'] = s.threshold_vol_dates_mann_kendall_test